# Exporta.co — Pipeline de Datos
**Maestría en Ciencia de Datos | Visualización y Storytelling con Datos**  
Responsable: Camilo José Beltrán Rojas | Apoyo: Juan Camilo Piñeros, Edgar Ivan Morales Rojas

Estructura modular. Cada sección puede ejecutarse independientemente una vez completadas las anteriores.  
Para migrar a Colab: cambiar `BASE_DIR` en la Sección 1 y subir los ZIPs.

---
## Sección 1 — Rutas
Unico bloque que cambia entre ejecución local y Colab.

In [ ]:
from pathlib import Path

BASE_DIR   = Path(r'C:\Users\camil\OneDrive\Documentos\MIAD\Visualización y storytelling\Proyecto')
OUTPUT_DIR = BASE_DIR / 'json_output'

# Colab:
# BASE_DIR   = Path('/content/')
# OUTPUT_DIR = Path('/content/json_output/')

OUTPUT_DIR.mkdir(exist_ok=True)
assert BASE_DIR.exists(), f'Carpeta no encontrada: {BASE_DIR}'
print(BASE_DIR)
print(OUTPUT_DIR)

---
## Sección 2 — Imports

In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec('pycountry') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pycountry', '-q'])

import pandas as pd
import numpy as np
import json, zipfile, io, re
from pathlib import Path
from collections import defaultdict

print(f'pandas {pd.__version__} | numpy {np.__version__}')

---
## Sección 3 — Configuración
Centraliza parámetros, mapas de columnas y diccionarios de referencia.  
Si el DANE cambia nombres de columnas en un año nuevo, solo editar aquí.

In [ ]:
# Rango de años a procesar
AÑOS = list(range(2011, 2026))

# Mapeo columnas EXPO: nombre real en CSV -> nombre interno del pipeline
COLUMNAS_EXPO = {
    'ï»¿fech': 'fech', 'fech': 'fech',
    'cod_pai4': 'pais',          # ya viene en ISO alfa-3 (USA, DEU...)
    'posar':    'subpartida',    # posición arancelaria HS
    'fobdol':   'fob_usd',
    'pnk':      'kg_neto',
}

# Mapeo columnas IMPO
COLUMNAS_IMPO = {
    'ï»¿fech': 'fech', 'fech': 'fech',
    'paisgen':  'pais',          # código numérico DIAN
    'naban':    'subpartida',
    'vacid':    'cif_usd',       # valor CIF dólares
    'pnk':      'kg_neto',
}

# Códigos numéricos DIAN -> ISO alfa-3 (fuente: diccionario de datos DIAN)
CODIGOS_PAIS_IMPO = {
    13:'AFG', 17:'ALB', 23:'DEU', 26:'ARM', 27:'ABW', 37:'AND', 40:'AGO',
    43:'ATG', 47:'ANT', 53:'SAU', 59:'DZA', 63:'ARG', 69:'AUS', 72:'AUT',
    74:'AZE', 77:'BHS', 80:'BHR', 81:'BGD', 83:'BRB', 87:'BEL', 88:'BLZ',
    90:'BMU', 91:'BLR', 93:'MMR', 97:'BOL', 101:'BWA', 105:'BRA', 108:'BRN',
    111:'BGR', 115:'BDI', 119:'BTN', 127:'CPV', 137:'CYM', 141:'KHM',
    145:'CMR', 149:'CAN', 169:'COL', 173:'COM', 177:'COG', 187:'PRK',
    190:'KOR', 193:'CIV', 196:'CRI', 198:'HRV', 199:'CUB', 203:'TCD',
    211:'CHL', 215:'CHN', 218:'TWN', 221:'CYP', 229:'BEN', 232:'DNK',
    235:'DMA', 239:'ECU', 240:'EGY', 242:'SLV', 243:'ERI', 244:'ARE',
    245:'ESP', 246:'SVK', 247:'SVN', 249:'USA', 251:'EST', 253:'ETH',
    267:'PHL', 271:'FIN', 275:'FRA', 281:'GAB', 285:'GMB', 287:'GEO',
    289:'GHA', 297:'GRD', 301:'GRC', 317:'GTM', 329:'GIN', 331:'GNQ',
    334:'GNB', 337:'GUY', 341:'HTI', 345:'HND', 351:'HKG', 355:'HUN',
    361:'IND', 365:'IDN', 369:'IRQ', 372:'IRN', 375:'IRL', 379:'ISL',
    383:'ISR', 386:'ITA', 391:'JAM', 399:'JPN', 403:'JOR', 406:'KAZ',
    410:'KEN', 412:'KGZ', 413:'KWT', 420:'LAO', 426:'LSO', 429:'LVA',
    431:'LBN', 434:'LBR', 438:'LBY', 440:'LIE', 443:'LTU', 445:'LUX',
    447:'MAC', 450:'MDG', 455:'MYS', 458:'MWI', 461:'MDV', 464:'MLI',
    467:'MLT', 474:'MAR', 477:'MTQ', 485:'MUS', 488:'MRT', 493:'MEX',
    496:'MDA', 497:'MNG', 498:'MCO', 505:'MOZ', 507:'NAM', 517:'NPL',
    521:'NIC', 525:'NER', 528:'NGA', 538:'NOR', 542:'NCL', 545:'PNG',
    548:'NZL', 551:'VUT', 556:'OMN', 573:'NLD', 576:'PAK', 580:'PAN',
    586:'PRY', 589:'PER', 603:'POL', 607:'PRT', 611:'PRI', 618:'QAT',
    628:'GBR', 640:'CAF', 644:'CZE', 647:'DOM', 660:'REU', 665:'ZWE',
    670:'ROU', 675:'RWA', 676:'RUS', 685:'ESH', 687:'WSM', 695:'KNA',
    697:'SMR', 705:'VCT', 715:'LCA', 720:'STP', 728:'SEN', 729:'SRB',
    731:'SYC', 735:'SLE', 741:'SGP', 744:'SYR', 748:'SOM', 750:'LKA',
    756:'ZAF', 759:'SDN', 764:'SWE', 767:'CHE', 770:'SUR', 774:'TJK',
    776:'THA', 780:'TZA', 783:'DJI', 788:'TLS', 800:'TGO', 810:'TON',
    815:'TTO', 820:'TUN', 825:'TKM', 827:'TUR', 830:'UKR', 833:'UGA',
    845:'URY', 847:'UZB', 850:'VEN', 855:'VNM', 863:'VGB', 866:'VIR',
    870:'FJI', 880:'YEM', 888:'COD', 890:'ZMB',
    # Zonas francas (911-928) y no declarados -> excluir
    **{c: None for c in range(911, 930)}, 999: None,
}

# Capítulos HS -> sector OMC (para derivar sector desde posición arancelaria)
# 01-24: Agropecuarios | 25-27: Combustibles | 28-97: Manufacturas
_SECTOR_BREAKS = [0, 24, 27, 97]
_SECTOR_LABELS = [
    'Agropecuarios, alimentos y bebidas',
    'Combustibles e industrias extractivas',
    'Manufacturas',
]

# Mes en español -> número (para extraer mes desde nombre de archivo IMPO)
MESES_ES = {
    'enero':1, 'febrero':2, 'marzo':3, 'abril':4,
    'mayo':5, 'junio':6, 'julio':7, 'agosto':8,
    'septiembre':9, 'octubre':10, 'noviembre':11, 'diciembre':12,
}

print('Configuracion cargada')
print(f'Anos: {AÑOS[0]}-{AÑOS[-1]}')

---
## Sección 4 — Descubrimiento de archivos
Escanea BASE_DIR y agrupa ZIPs por tipo y año.  
Maneja años partidos en sufijos _1 y _2 automáticamente.

In [ ]:
PATRON = re.compile(r'^(expo|impo)_(\d{4})(?:_(\d))?\.zip$', re.IGNORECASE)
archivos = {'expo': defaultdict(list), 'impo': defaultdict(list)}

for f in sorted(BASE_DIR.glob('*.zip')):
    m = PATRON.match(f.name)
    if m:
        tipo = m.group(1).lower()
        año  = int(m.group(2))
        archivos[tipo][año].append(f)

for tipo in ('expo', 'impo'):
    años_rango   = sorted(a for a in archivos[tipo] if a in AÑOS)
    años_fuera   = sorted(a for a in archivos[tipo] if a not in AÑOS)
    años_faltant = [a for a in AÑOS if a not in archivos[tipo]]
    print(f'\n{tipo.upper()} -- {len(años_rango)} año(s) en rango')
    for año in años_rango:
        partes = archivos[tipo][año]
        tag = f'[{len(partes)} partes]' if len(partes) > 1 else ''
        for p in partes:
            print(f'  {p.name} {tag}')
    if años_faltant:
        print(f'  Faltantes en rango: {años_faltant}')
    if años_fuera:
        print(f'  Fuera de rango (ignorados): {años_fuera}')

---
## Sección 5 — Funciones de ingesta
Funciones separadas para EXPO e IMPO dado que difieren en:
- Formato de fecha: EXPO usa YYMM, IMPO usa YYYY (mes se extrae del nombre del archivo)
- Identificador de país: EXPO trae ISO alfa-3 directo, IMPO trae código numérico DIAN

In [ ]:
def normalizar_columnas(df, mapa):
    rename = {}
    for col in df.columns:
        clave = col.strip().lower().replace(' ', '_')
        if clave in mapa:
            rename[col] = mapa[clave]
        elif col.strip() in mapa:
            rename[col] = mapa[col.strip()]
    return df.rename(columns=rename)


def limpiar_valor_monetario(serie):
    return (
        serie.astype(str)
        .str.replace(',', '', regex=False)
        .str.replace(' ', '', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )


def derivar_sector(serie_subpartida):
    """Deriva sector OMC desde los 2 primeros dígitos de la posición arancelaria (capítulo HS)."""
    capitulo = pd.to_numeric(serie_subpartida.astype(str).str[:2], errors='coerce')
    return (
        pd.cut(capitulo, bins=_SECTOR_BREAKS, labels=_SECTOR_LABELS)
        .astype(str)
        .replace('nan', 'Otros sectores')
    )


def mes_desde_nombre(nombre_archivo):
    """Extrae número de mes desde nombre de archivo (ej: 'Agosto 2024/Agosto.csv' -> 8)."""
    stem = Path(nombre_archivo).stem.lower().strip()
    return MESES_ES.get(stem, None)


def leer_csv_en_memoria(data_bytes):
    """Lee CSV desde bytes probando encodings y separadores comunes del DANE."""
    for encoding in ('latin-1', 'utf-8', 'utf-8-sig'):
        for sep in (',', ';', '\t'):
            try:
                df = pd.read_csv(
                    io.BytesIO(data_bytes),
                    sep=sep,
                    encoding=encoding,
                    low_memory=False,
                    on_bad_lines='skip',
                )
                if len(df.columns) > 2:
                    return df.dropna(how='all')
            except Exception:
                continue
    return None


def leer_zip_expo(ruta_zip):
    """
    Estructura: ZIP anual -> ZIPs mensuales -> CSV
    Año y mes se derivan de FECH (formato YYMM).
    """
    frames = []
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            zips_mes = sorted(f for f in zf.namelist() if f.lower().endswith('.zip'))
            for nom_zip in zips_mes:
                try:
                    with zipfile.ZipFile(io.BytesIO(zf.read(nom_zip)), 'r') as zf_mes:
                        for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                            df = leer_csv_en_memoria(zf_mes.read(csv_path))
                            if df is not None:
                                frames.append(normalizar_columnas(df, COLUMNAS_EXPO))
                except Exception as e:
                    print(f'    Advertencia {nom_zip}: {e}')
    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


def leer_zip_impo(ruta_zip):
    """
    Estructura: ZIP anual -> ZIPs mensuales -> CSV
    Año desde FECH (YYYY), mes desde nombre del archivo CSV.
    """
    frames = []
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            zips_mes = sorted(f for f in zf.namelist() if f.lower().endswith('.zip'))
            for nom_zip in zips_mes:
                try:
                    with zipfile.ZipFile(io.BytesIO(zf.read(nom_zip)), 'r') as zf_mes:
                        for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                            df = leer_csv_en_memoria(zf_mes.read(csv_path))
                            if df is not None:
                                df = normalizar_columnas(df, COLUMNAS_IMPO)
                                mes = mes_desde_nombre(csv_path)
                                if mes:
                                    df['mes'] = mes
                                frames.append(df)
                except Exception as e:
                    print(f'    Advertencia {nom_zip}: {e}')
    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


print('Funciones listas')

---
## Sección 6 — Ingesta
Para exploración, limitar AÑOS en la Sección 3 a un solo año antes de correr todo.

In [ ]:
def _ingestar(tipo, fn_leer):
    frames = []
    for año in AÑOS:
        partes = archivos[tipo].get(año, [])
        if not partes:
            print(f'  {tipo.upper()} {año}: no encontrado, omitido')
            continue
        frames_año = []
        for ruta in sorted(partes):
            frames_año.extend(fn_leer(ruta))
        if frames_año:
            df = pd.concat(frames_año, ignore_index=True)
            df['año'] = año
            frames.append(df)
            print(f'  {tipo.upper()} {año}: {len(df):,} filas')
        else:
            print(f'  {tipo.upper()} {año}: sin datos legibles')
    return pd.concat(frames, ignore_index=True) if frames else None


df_expo = _ingestar('expo', leer_zip_expo)
df_impo = _ingestar('impo', leer_zip_impo)

---
## Sección 7 — Diagnostico de columnas
Verificar antes de limpiar. Si hay columnas faltantes, ajustar los mapas en la Sección 3
y re-ejecutar desde la Sección 6.

In [ ]:
COLS_EXPO = ['fech', 'pais', 'subpartida', 'fob_usd', 'kg_neto', 'mes']
COLS_IMPO = ['fech', 'pais', 'subpartida', 'cif_usd', 'kg_neto', 'mes']

def diagnostico(df, nombre, cols):
    if df is None:
        print(f'{nombre}: DataFrame vacio'); return
    print(f'\n{nombre}')
    print(f'  Filas    : {len(df):,}')
    print(f'  Anos     : {sorted(df["año"].dropna().unique().astype(int).tolist())}')
    print(f'  Columnas : {list(df.columns)}')
    for c in cols:
        estado = 'OK' if c in df.columns else 'FALTA'
        print(f'  [{estado}] {c}')

diagnostico(df_expo, 'EXPORTACIONES', COLS_EXPO)
diagnostico(df_impo, 'IMPORTACIONES', COLS_IMPO)

---
## Sección 8 — Limpieza
Ejecutar solo cuando el diagnostico muestre OK en todas las columnas requeridas.

In [ ]:
# EXPORTACIONES
print('Limpiando EXPORTACIONES...')
n0 = len(df_expo)

fech = pd.to_numeric(df_expo['fech'], errors='coerce')
df_expo['año'] = (2000 + fech // 100).astype('Int64')
df_expo['mes'] = (fech % 100).astype('Int64')

df_expo['iso3']    = df_expo['pais'].str.strip().str.upper()
df_expo['fob_usd'] = limpiar_valor_monetario(df_expo['fob_usd'])
df_expo['sector']  = derivar_sector(df_expo['subpartida'])

df_expo = df_expo[
    df_expo['fob_usd'].notna() & (df_expo['fob_usd'] > 0) &
    df_expo['iso3'].notna() & (df_expo['iso3'] != '') &
    df_expo['año'].between(AÑOS[0], AÑOS[-1]) &
    df_expo['mes'].between(1, 12)
]
print(f'  Eliminadas: {n0 - len(df_expo):,} | Restantes: {len(df_expo):,}')
print(df_expo[['año','mes','iso3','fob_usd','sector']].head(3))

In [ ]:
# IMPORTACIONES
print('Limpiando IMPORTACIONES...')
n0 = len(df_impo)

df_impo['año'] = pd.to_numeric(df_impo['fech'], errors='coerce').astype('Int64')
df_impo['mes'] = pd.to_numeric(df_impo['mes'],  errors='coerce').astype('Int64')

codigo = pd.to_numeric(df_impo['pais'], errors='coerce')
df_impo['iso3']    = codigo.map(CODIGOS_PAIS_IMPO)
df_impo['cif_usd'] = limpiar_valor_monetario(df_impo['cif_usd'])

df_impo = df_impo[
    df_impo['cif_usd'].notna() & (df_impo['cif_usd'] > 0) &
    df_impo['iso3'].notna() &
    df_impo['año'].between(AÑOS[0], AÑOS[-1]) &
    df_impo['mes'].between(1, 12)
]
print(f'  Eliminadas: {n0 - len(df_impo):,} | Restantes: {len(df_impo):,}')
print(df_impo[['año','mes','iso3','cif_usd']].head(3))

---
## Sección 9 — Exportacion de JSON
*Modulo pendiente. Se agrega una vez validada la limpieza.*